# Prompt-aware attribution: GPU research experiment

**Research only — this notebook does not create a submission.** It compares classification, supervised contrastive learning, and same-prompt hard-negative weighting using prompt-grouped CV on the selection partition. Reserved prompts are not scored. Since historical experiments already used the full dataset, the reserved partition is not a pristine independent holdout.

Setup: enable a Kaggle GPU and attach (1) the competition data, (2) a Kaggle dataset containing this repository's `src/`, `experiments/`, and `configs/`, and (3) a publicly accessible Hugging Face encoder checkpoint with its tokenizer (for example DeBERTa-v3-base). Verify checkpoint license and competition rules. Internet may remain **off**; the checkpoint and code must be attached locally. Torch, transformers, pandas, numpy, scipy, scikit-learn and PyYAML must be available in the notebook image. Set the exact paths below after attaching inputs.

This executes 5 folds × 3 variants; runtime and GPU memory depend on the model and GPU. Start with a copy of the YAML configured for a small pilot if needed, but do not compare pilot metrics to a full CV score. Outputs may contain competition row IDs and labels: treat them as private competition artifacts, not public datasets. No hidden test inputs are read.


In [ ]:
from pathlib import Path
import torch

# CHANGE THESE TWO PATHS to match your attached Kaggle input datasets.
REPO_DIR = Path('/kaggle/input/YOUR-REPOSITORY-DATASET')
CHECKPOINT = Path('/kaggle/input/YOUR-ENCODER-DATASET/model')
TRAIN = Path('/kaggle/input/competitions/name-that-llm/train.csv')
CONFIG = REPO_DIR / 'configs/contrastive_gpu.yaml'
OUTPUT = Path('/kaggle/working/contrastive_gpu_run')  # must not already exist

assert torch.cuda.is_available(), 'Enable GPU in Kaggle notebook settings.'
assert (REPO_DIR / 'experiments/contrastive_gpu.py').is_file(), 'Attach the source dataset and set REPO_DIR.'
assert CHECKPOINT.is_dir(), 'Attach the encoder checkpoint and set CHECKPOINT.'
assert TRAIN.is_file(), 'Attach the competition input and verify TRAIN.'
assert CONFIG.is_file()
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
import subprocess
import sys

# Offline guarantees: no checkpoint downloads and no network calls by this run.
env = {**os.environ, 'HF_HUB_OFFLINE': '1', 'TRANSFORMERS_OFFLINE': '1'}
command = [sys.executable, str(REPO_DIR / 'experiments/contrastive_gpu.py'),
           '--train', str(TRAIN), '--checkpoint', str(CHECKPOINT),
           '--config', str(CONFIG), '--output', str(OUTPUT)]
subprocess.run(command, env=env, check=True, cwd=REPO_DIR)


In [ ]:
import json
metrics = json.loads((OUTPUT / 'metrics.json').read_text())
for name, result in metrics.items():
    print(f"{name}: grouped CV balanced accuracy = {result['balanced_accuracy']:.4f}")
print('Research results only; do not treat these as a leaderboard score.')
